# 01. Quickstart: PBMC CLI & Config-Driven Pipeline

Welcome to **FlowX**! In this 5-minute quickstart tutorial, you will run a complete, reproducible flow cytometry analysis on a classic **human PBMC** dataset (`finak_2014_cytotrol_pbmc` — 2 Cyto-Trol lyophilized PBMC standards, 7 fluorescence channels, ~18s runtime).

**What you will learn:**
1. How to inspect raw `.fcs` headers and curated cohort metadata (`samples.csv`, `markers.csv`).
2. How to execute the unified config-driven pipeline (`flowx pipeline -c ...`).
3. How to audit `resolved_config.yaml` and inspect the discovered PBMC populations (`CD4+`, `CD8+`, `HLA-DR+ APC / B-Cell / Monocyte`, `NK / CD3-`).
4. Where FlowX writes Gating-ML 2.0, Augmented FCS, and MIFlowCyt reports.

> **Prerequisites (Download Raw `.fcs` Data):** Curated `metadata/samples.csv` and `metadata/markers.csv` are included in the repository. Before running this notebook, download the raw `.fcs` files once from the terminal:
> ```bash
> flowx download finak_2014_cytotrol_pbmc
> ```

In [1]:
import os
from pathlib import Path

# Ensure working directory is the repository root whether launched from root or docs/tutorials/
if not Path("configs/config_run").exists() and Path("../../configs/config_run").exists():
    os.chdir("../..")
print("Working directory ready:", Path("configs/config_run").exists())

Working directory ready: True


## 1. Inspect Curated Metadata & Config

FlowX uses `metadata/samples.csv` (cohort design) and `metadata/markers.csv` (panel roles) so your YAML config can use `"auto"` instead of hardcoding channel lists.

In [2]:
import pandas as pd
from pathlib import Path

samples_df = pd.read_csv("data/conventional/finak_2014_cytotrol_pbmc/metadata/samples.csv")
markers_df = pd.read_csv("data/conventional/finak_2014_cytotrol_pbmc/metadata/markers.csv")

print("=== samples.csv ===")
print(samples_df[["sample_id", "condition", "batch_id", "is_control", "total_events"]].to_string(index=False))
print("\n=== markers.csv (fluorescence channels) ===")
fl = markers_df[~markers_df["marker_class"].isin(["scatter", "time"])]
print(fl[["channel_name", "marker_name", "fluorophore", "marker_class"]].to_string(index=False))

=== samples.csv ===
          sample_id              condition         batch_id  is_control  total_events
CytoTrol_CytoTrol_1 CytoTrol_PBMC_Standard 2012-06-05_LSRII       False        119531
CytoTrol_CytoTrol_2 CytoTrol_PBMC_Standard 2012-06-05_LSRII       False        115728

=== markers.csv (fluorescence channels) ===
channel_name marker_name fluorophore       marker_class
      B710-A         CD4 PerCP-Cy5.5            lineage
      R660-A        CD38         APC         functional
      R780-A         CD8      APC-H7            lineage
      V450-A         CD3        V450            lineage
      V545-A      HLA-DR        V500 lineage;functional
      G560-A        CCR7          PE         functional
      G780-A      CD45RA      PE-Cy7         functional


## 2. Run Pre-Flight Inspection & Full Unified Pipeline

You can run both commands directly from your terminal (or inside a notebook cell with `!`).

```bash
# Optional pre-flight header audit
flowx inspect data/conventional/finak_2014_cytotrol_pbmc/raw/samples -o outputs/tutorial_inspect

# Run the full 6-step unified pipeline (~18 seconds)
flowx pipeline -c configs/config_run/conventional/finak_2014_cytotrol_pbmc.yaml --overwrite
```

You can also invoke the exact same unified pipeline programmatically via `PipelineConfig`:

In [3]:
from flowx.io.config import PipelineConfig

cfg = PipelineConfig.load("configs/config_run/conventional/finak_2014_cytotrol_pbmc.yaml")
print("Dataset output_dir:", cfg.output_dir)
print("Transform:", cfg.transform_type, "cofactor=", cfg.transform_cofactor)
print("Discovery:", cfg.clustering_method, "auto_k=", cfg.auto_k, "method=", cfg.auto_k_method)

# To execute the full pipeline from Python, uncomment:
# pipeline, input_path = cfg.to_pipeline()
# result = pipeline.run(input_path)

Dataset output_dir: outputs/config_run/conventional/finak_2014_cytotrol_pbmc
Transform: arcsinh cofactor= 150.0
Discovery: som auto_k= True method= consensus


## 3. Audit `resolved_config.yaml` & Cleanup Retention

Every `flowx pipeline` run writes `<output_dir>/resolved_config.yaml` recording the exact resolved scatter channels, transformed channels, and lineage markers.

In [4]:
import yaml

out_dir = Path("outputs/config_run/conventional/finak_2014_cytotrol_pbmc")
resolved = yaml.safe_load((out_dir / "resolved_config.yaml").read_text())
print("Resolved lineage markers:", resolved.get("05_population_discovery", {}).get("lineage_markers"))

cleanup_df = pd.read_csv(out_dir / "04_batch_preparation/tables/cohort_batch_summary.csv")
print("\n=== Step 03 -> 04 Event Retention Summary ===")
print(cleanup_df[["sample_id", "raw_event_count", "clean_event_count", "prepared_event_count", "retention_pct_of_raw"]].to_string(index=False))

Resolved lineage markers: ['B710-A', 'R780-A', 'V450-A', 'V545-A']

=== Step 03 -> 04 Event Retention Summary ===
          sample_id  raw_event_count  clean_event_count  prepared_event_count  retention_pct_of_raw
CytoTrol_CytoTrol_1           119531              86858                 50000             41.830153
CytoTrol_CytoTrol_2           115728              90774                 50000             43.204756


![PBMC Cleanup Cascade (Cyto-Trol)](../images/tutorials/cleanup_pbmc_finak.png)

## 4. Inspect Discovered PBMC Populations & UMAP

FlowX trains a $10 \times 10$ Self-Organizing Map (MiniSom), selects optimal $k=13$ via Consensus PAC, and maps all 13 clusters with **0 unassigned clusters** into 6 canonical human PBMC lineages:

In [5]:
pop_map = pd.read_csv(out_dir / "05_population_discovery/tables/cluster_to_population_map.csv")
print(pop_map.to_string(index=False))

 Cluster_ID Cluster_Label                    Annotated_Population                    Macro_Group  Event_Count  Pct_Total_Events
          0            C0                     CD4+ T-Cell Lineage            CD4+ T-Cell Lineage        15351            15.351
          1            C1 HLA-DR+ APC / Monocyte / B-Cell Lineage HLA-DR+ APC / Monocyte Lineage         6255             6.255
          2            C2                     CD8+ T-Cell Lineage            CD8+ T-Cell Lineage        17130            17.130
          3            C3                    CD4+ CD8+ DP T Cells           CD4+ CD8+ DP T Cells         2048             2.048
          4            C4                  CD3- Non-T Lymphocytes         CD3- Non-T Lymphocytes         4333             4.333
          5            C5                 CD3+ DN / Other T Cells        CD3+ DN / Other T Cells         1020             1.020
          6            C6                  CD3- Non-T Lymphocytes         CD3- Non-T Lymphocytes        

![PBMC UMAP Populations (Cyto-Trol)](../images/tutorials/discovery_umap_pop_pbmc_finak.png)

## 5. Explore Interoperable Exports

FlowX automatically generates ISAC Gating-ML 2.0 XML, Augmented `.fcs` files (with numeric `cluster_id`, `UMAP_1`, `UMAP_2` channels and linked biological `lineage_id` gates), and MIFlowCyt HTML/Markdown reports:

In [6]:
for p in sorted((out_dir / "exports").rglob("*")):
    if p.is_file():
        print(p.relative_to(out_dir))

exports/augmented_fcs/augmented_CytoTrol_CytoTrol_1.fcs
exports/augmented_fcs/augmented_CytoTrol_CytoTrol_2.fcs
exports/reports/miflowcyt_analysis_report.html
exports/reports/miflowcyt_analysis_report.md
exports/reports/report.md
exports/workspaces/flowjo_workspace.wsp
exports/workspaces/gating_ml_2.0.xml
exports/workspaces/preprocessing_gating.wsp
exports/workspaces/preprocessing_gating_ml_2.0.xml
exports/workspaces/workspace.json
exports/workspaces/workspace.xml


## Next Steps

- Continue to **[02. Conventional PBMC Cohort Walkthrough](02_conventional_pbmc_cohort_walkthrough.ipynb)** for multi-batch longitudinal PBMC analysis, single-stain control spillover calculation, and Differential Abundance/State testing.

## References & Scientific Standards

* **MIFlowCyt Standard:** [](doi:10.1002/cyto.a.20623)
* **FCS 3.1 Data File Standard:** [](doi:10.1002/cyto.a.20825)
* **Gating-ML 2.0 Standard:** [](doi:10.1002/cyto.a.22690)
* **FlowSOM Clustering:** [](doi:10.1002/cyto.a.22625)
* **CytoNorm Batch Normalization:** [](doi:10.1002/cyto.a.23904)
* **UMAP Dimensionality Reduction:** [](doi:10.48550/arXiv.1802.03426)
* **FlowKit Computational Toolkit:** [](doi:10.3389/fimmu.2021.768541)